# ser_client-api + ser_client-ftps - Demo against ser_server-ftps

Demonstrates the client-side stack against the mock FTPS server:

1. **ser_client-api**: parse a GLeaves prescription JSON into domain model objects (`CompositionData`), generate an HL7v2 ORU_R01 message, and parse the ACK response
2. **ser_client-ftps**: transfer files to `ser_server-ftps` and retrieve the ACK file

**Prerequisites**

- `ser_server-ftps` repository checked out
- Client certificates: `client-cert.pem`, `client-key.pem`, `ca-cert.pem`
- Both packages installed:
  ```bash
  pip install git+https://github.com/GenomeCAD/ser_client-api.git@v0.2.0-rc2
  pip install git+https://github.com/GenomeCAD/ser_client-ftps.git@v0.3.0-rc1
  ```

**Start the mock server**

In the `ser_server-ftps` root directory:
```bash
docker compose up -d
```

Mock server: `localhost:990`, credentials: `CHU-TEST` / `ftppassword`

## 0. Configuration

In [ ]:
from pathlib import Path

FTPS_HOST = "localhost"
FTPS_PORT = 990
FTPS_USER = "CHU-TEST"
FTPS_PASSWORD = "ftppassword"

# Remote base path on the FTPS server (institution.path in the database).
# Files are uploaded to {REMOTE_PATH}/{PRESCRIPTION_NAME}/ and the ACK is pulled from the same folder.
REMOTE_PATH = "remote/seqoia"

In [ ]:
from ser_client_api.hl7v2 import HL7v2Generator, InstitutionConfig

# Compiled GIPCAD profiles; set to the actual paths on your system
HL7_PROFILE_PATH = Path("/path/to/profiles/oru_r01_lab36")
ACK_PROFILE_PATH = Path("/path/to/profiles/ack_r01_ack")

# Client certificates for FTPS; set to the actual path on your system
CERT_DIR = Path("/path/to/certificates")

# Sending institution identifiers - override with your own institution's values
institution = InstitutionConfig(
    lab_name="GCS LBM SEQOIA SITE BROUSSAIS",
    lab_finess="6750063265",
    facility_name="GCS SEQOIA",
    facility_finess="5750059800",
    facility_finess_ej="750059800",
    software_name="gleaves",
    software_product_information="variantannotator^1.2.250.1.710.1.7.3.2.9^1.2.250.1.710.1.2.1",
    receiving_application="GIP COLLECTEUR ANALYSEUR DE DONNEES^313003057000027^1.2.250.1.71.4.2.2",
    receiving_facility="GIP COLLECTEUR ANALYSEUR DE DONNEES^2130030570^1.2.250.1.71.4.2.2",
)

## 1. Parse prescription JSON

In [ ]:
# Minimal prescription payload in GLeaves format, trio case (patient + father + mother).

# To read from a JSON file instead, replace the PRESCRIPTION_JSON definition below with:
# PRESCRIPTION_JSON = json.loads(Path("/path/to/prescription.json").read_text())

PRESCRIPTION_JSON = {
    "_id": "67cecdfdb4cffa55e8707002",
    "preindication": {
        "catname": "Maladie rare",
        "catkey": "p1",
        "name": "Angiodèmes bradykiniques héréditaires",
        "key": "p1-sp60",
        "rcp_id": "5e71f8b3efaa9b6f5a729fa6",
        "rcp_nom": "MaRIH-NEUTROPENIES",
    },
    "patients": [
        {
            "patient": {
                "id": {"type": "IPP", "value": "IPP-DEMO-001"},
                "date_naissance": "1980-06-15",
                "sexe": "F",
                "nom": "DUPONT",
                "nom_naissance": "DUPONT",
                "prenom": "MARIE",
                "date_prelevement": 1741651200000,
            },
            "lien": {"key": "patient", "name": "Patient"},
            "is_data_reusable_for_research": True,
            "dateConsent": 1741940668351,
            "id_anon": "66EFA7",
            "resultat_compte_rendu_gleaves": {},
        },
        {
            "patient": {
                "id": {"type": "IPP", "value": "IPP-DEMO-002"},
                "date_naissance": "1955-03-22",
                "sexe": "M",
                "nom": "DUPONT",
                "nom_naissance": "DUPONT",
                "prenom": "JEAN",
            },
            "lien": {"key": "père", "name": "Père"},
            "is_data_reusable_for_research": False,
            "date_prelevement": 1741651200000,
            "id_anon": "66EFA8",
        },
        {
            "patient": {
                "id": {"type": "IPP", "value": "IPP-DEMO-003"},
                "date_naissance": "1958-11-04",
                "sexe": "F",
                "nom": "MARTIN",
                "nom_naissance": "MARTIN",
                "prenom": "CLAIRE",
            },
            "lien": {"key": "mère", "name": "Mère"},
            "is_data_reusable_for_research": False,
            "date_prelevement": 1741651200000,
            "id_anon": "66EFA9",
        },
    ],
    "prescripteur": "4031575",
    "membreRCP": "541351",
    "analysis_info": {
        "analysis_ID": "55014",
        "date_fin_analyse": "08042025",
    },
    "date_creation": 1741606397865,
    "date_cloture": 1745498322775,
    "resultats": [
        {
            "type": "cr_biologique",
            "MembreLMG": "541351",
            "filename": "CR_SeqOIA_Demo.pdf",
        }
    ],
}

In [ ]:
from ser_client_api.hl7v2.domain_models import CompositionData
from ser_client_api.hl7v2.gleaves import GleavesJSONParser as PrescriptionParser

parser = PrescriptionParser()
composition: CompositionData = parser.parse(PRESCRIPTION_JSON)

print(f"Report ID    : {composition.report_id}")
print(f"Patient      : {composition.patient.patient_family_name} {composition.patient.patient_given_name}")
print(f"DOB          : {composition.patient.birth_date}  sex={composition.patient.sex}")
print(f"Preindication: {composition.preindication.name} ({composition.preindication.key})")
print(f"RCP          : {composition.rcp.rcp_nom} (id={composition.rcp.rcp_id})")
print(f"Analysis ID  : {composition.analysis.analysis_id}")
print(f"Period       : {composition.timing.start} > {composition.timing.end}")
print(f"Prescripteur : {composition.person.prescripteur}")
print(f"MembreLMG    : {composition.results.membre_lmg}")
print(f"Consent      : reusable={composition.consent.is_data_reusable_for_research}")
if composition.next_of_kin:
    for nok in composition.next_of_kin:
        print(f"Next of kin  : [{nok.relationship_code}] {nok.family_name} {nok.given_name} (DOB: {nok.birth_date})")

## 2. Prepare files to transfer

In production, genomic data files may come from big data storage infrastructures (downloaded from S3, restored from tape data storage...).

For this demo, small synthetic files are created with a given directory structure.

In [ ]:
import hashlib
import tempfile

PRESCRIPTION_NAME = "DEMO-PRESCRIPTION-001"

tmp_dir = Path(tempfile.mkdtemp(prefix="ser_demo_"))
presc_dir = tmp_dir / PRESCRIPTION_NAME

# Main patient files (id_anon = 66EFA7)
patient_dir = presc_dir / "66EFA7"
patient_dir.mkdir(parents=True)
(patient_dir / f"{PRESCRIPTION_NAME}_final.vcf").write_text("##fileformat=VCFv4.2\n#CHROM\tPOS\tID\tREF\tALT\n")
(patient_dir / f"{PRESCRIPTION_NAME}_final.tar.gz").write_bytes(b"\x1f\x8b" + b"\x00" * 20)
(patient_dir / f"{PRESCRIPTION_NAME}_chr1_markdup.bam").write_bytes(b"BAM\x01" + b"\x00" * 100)
(patient_dir / f"{PRESCRIPTION_NAME}_chr1_markdup.bam.bai").write_bytes(b"BAI\x01" + b"\x00" * 20)

# Per-file .sha256 sidecars - required by ser_server-ftps for integrity verification
def write_sha256(filepath: Path):
    digest = hashlib.sha256(filepath.read_bytes()).hexdigest()
    (filepath.parent / (filepath.name + ".sha256")).write_text(f"{digest}  {filepath.name}\n")

for f in sorted(patient_dir.rglob("*")):
    if f.is_file():
        write_sha256(f)

all_files = [f for f in sorted(presc_dir.rglob("*")) if f.is_file()]
print(f"Transfer directory: {presc_dir}")
print(f"Files to transfer ({len(all_files)}):")
for f in all_files:
    print(f"  {f.relative_to(presc_dir)}  ({f.stat().st_size} bytes)")

## 3. Generate HL7v2 message

Builds an ORU_R01 message from the `CompositionData` and writes it to the transfer directory.  
OBX segments are populated from the files prepared above.

In [ ]:
generator = HL7v2Generator(profile_path=str(HL7_PROFILE_PATH), institution=institution)

hl7_message = generator.generate(composition, files_directory=str(presc_dir))

hl7_file = presc_dir / f"{PRESCRIPTION_NAME}.hl7"
hl7_file.write_text(hl7_message, encoding="utf-8")

# .hl7.sha256 - required by ser_server-ftps to verify the HL7 file itself
hl7_digest = hashlib.sha256(hl7_file.read_bytes()).hexdigest()
(presc_dir / f"{PRESCRIPTION_NAME}.hl7.sha256").write_text(f"{hl7_digest}  {hl7_file.name}\n")

segments = hl7_message.split("\r")
print(f"HL7v2 file: {hl7_file.name}  ({len(hl7_message)} bytes, {len(segments)} segments)")
print()
for seg in segments[:6]:
    print(seg[:100] + ("..." if len(seg) > 100 else ""))

## 4. Transfer files to ser_server-ftps

In [ ]:
import os
from ser_client_ftps import SecureFTPSClient
from ser_client_ftps.exceptions import FTPSConnectionError, FTPSTransferError

client = SecureFTPSClient(
    host=FTPS_HOST,
    port=FTPS_PORT,
    cert_file=str(CERT_DIR / "client-cert.pem"),
    key_file=str(CERT_DIR / "client-key.pem"),
    ca_file=str(CERT_DIR / "ca-cert.pem"),
)

REMOTE_DIR = f"{REMOTE_PATH}/{PRESCRIPTION_NAME}"
files_transferred = []

try:
    with client.connect(FTPS_USER, FTPS_PASSWORD) as host:
        print(f"Connected to {FTPS_HOST}:{FTPS_PORT}")

        if not host.path.exists(REMOTE_DIR):
            host.makedirs(REMOTE_DIR)
            print(f"Created remote directory: {REMOTE_DIR}")

        for local_file in sorted(presc_dir.rglob("*")):
            if not local_file.is_file():
                continue

            relative = local_file.relative_to(presc_dir)
            remote_file = f"{REMOTE_DIR}/{relative}"

            remote_subdir = os.path.dirname(remote_file)
            if remote_subdir and not host.path.exists(remote_subdir):
                host.makedirs(remote_subdir)

            host.upload(str(local_file), remote_file)

            remote_size = host.path.getsize(remote_file)
            local_size = local_file.stat().st_size
            assert remote_size == local_size, f"Size mismatch on {relative}"

            files_transferred.append(remote_file)
            print(f"  ok {relative}  ({remote_size} bytes)")

        # Upload .hl7.ok marker - triggers ACK generation on ser_server-ftps
        hl7_ok_local = presc_dir / f"{PRESCRIPTION_NAME}.hl7.ok"
        hl7_ok_local.write_text("")
        hl7_ok_remote = f"{REMOTE_DIR}/{PRESCRIPTION_NAME}.hl7.ok"
        host.upload(str(hl7_ok_local), hl7_ok_remote)
        print(f"  ok {PRESCRIPTION_NAME}.hl7.ok  (trigger)")

        print(f"\nDone: {len(files_transferred)} files transferred")

except FTPSConnectionError as e:
    print(f"Connection error (is ser_server-ftps running?): {e}")
except FTPSTransferError as e:
    print(f"Transfer error: {e}")

## 5. Verify - list transferred files on the server

In [ ]:
with client.connect(FTPS_USER, FTPS_PASSWORD) as host:
    print(f"Remote /{REMOTE_DIR}/:")
    for entry in sorted(host.listdir(REMOTE_DIR)):
        remote_path = f"{REMOTE_DIR}/{entry}"
        if host.path.isdir(remote_path):
            print(f"  {entry}/")
            for sub in sorted(host.listdir(remote_path)):
                size = host.path.getsize(f"{remote_path}/{sub}")
                print(f"    {sub}  ({size} bytes)")
        else:
            size = host.path.getsize(remote_path)
            print(f"  {entry}  ({size} bytes)")

## 6. Retrieve and parse ACK response

After the transfer, `ser_server-ftps` processes the HL7v2 message and writes an ACK file back to the remote folder.
We pull it via FTPS and parse it with `ser_client_api`.

In [ ]:
ack_filename, ack_content = client.pull_ack(
    username=FTPS_USER,
    password=FTPS_PASSWORD,
    remote_folder_path=REMOTE_DIR,
    institution_name="demo",
)

if ack_filename is None:
    raise RuntimeError("No ACK file found on server - has ser_server-ftps finished processing?")

print(f"Pulled ACK: {ack_filename}")

In [ ]:
from hl7apy import load_message_profile
from hl7apy.consts import VALIDATION_LEVEL
from hl7apy.parser import parse_message
from ser_client_api import _analyze_ack_message, _determine_transfer_status

ack_profile = load_message_profile(str(ACK_PROFILE_PATH))
normalized = ack_content.replace("\r\n", "\r").replace("\n", "\r")
ack_msg = parse_message(normalized, message_profile=ack_profile, validation_level=VALIDATION_LEVEL.STRICT)

analysis = _analyze_ack_message(ack_msg)
status = _determine_transfer_status(analysis)

status_label = {0: "OK", 1: "ERROR (retry)", 2: "FAILED"}.get(status, "UNKNOWN")

print(f"ACK file   : {ack_filename}")
print(f"MSA status : {analysis.msa_status}")
print(f"Control ID : {analysis.message_control_id}")
print(f"Errors     : {len(analysis.critical_errors)}")
print(f"Warnings   : {len(analysis.warnings)}")
print(f"Infos      : {len(analysis.infos)}")
print(f"Result     : {status_label}")
if analysis.critical_errors:
    print()
    for e in analysis.critical_errors:
        print(f"  error: {e}")
if analysis.warnings:
    print()
    for w in analysis.warnings:
        print(f"  warning: {w}")

## 7. Cleanup

In [ ]:
import shutil

shutil.rmtree(tmp_dir)
print(f"Removed local temp dir: {tmp_dir}")

### Optional: remove remote files

Run this before re-running the transfer cell to avoid `Overwrite permission denied` errors.

In [ ]:
with client.connect(FTPS_USER, FTPS_PASSWORD) as host:
    if not host.path.exists(REMOTE_DIR):
        print(f"Not found on server: {REMOTE_DIR}")
    else:
        for entry in host.listdir(REMOTE_DIR):
            remote_path = f"{REMOTE_DIR}/{entry}"
            if host.path.isdir(remote_path):
                for sub in host.listdir(remote_path):
                    host.remove(f"{remote_path}/{sub}")
                host.rmdir(remote_path)
            else:
                host.remove(remote_path)
        host.rmdir(REMOTE_DIR)
        print(f"Removed remote: {REMOTE_DIR}")